### Load the data and perform the merging

In [ ]:
import pandas as pd
headline_df = pd.read_csv("/content/drive/MyDrive/financial_data/finbert_emb_v2.csv", index_col = 0).rename(columns={'stock_name': 'stock'})
ohlcv_df = pd.read_csv("/content/drive/MyDrive/financial_data/ohlcv.csv", index_col = 0)
is_easy_df = pd.read_csv("/content/drive/MyDrive/financial_data/is_easy_1_5_20_v2.csv", index_col = 0)[['stock', 'date', 'easy_indicator']]

In [ ]:
headline_df

,Date,stock,file_path
0,2009-08-07,AXP,/content/drive/MyDrive/financial_data/finbert_...
1,2009-08-07,MA,/content/drive/MyDrive/financial_data/finbert_...
2,2009-08-10,AXP,/content/drive/MyDrive/financial_data/finbert_...
3,2009-08-10,COP,/content/drive/MyDrive/financial_data/finbert_...
4,2009-08-10,HD,/content/drive/MyDrive/financial_data/finbert_...
...,...,...,...
26588,2020-07-14,general,/content/drive/MyDrive/financial_data/finbert_...
26589,2020-07-15,general,/content/drive/MyDrive/financial_data/finbert_...
26590,2020-07-16,general,/content/drive/MyDrive/financial_data/finbert_...
26591,2020-07-17,general,/content/drive/MyDrive/financial_data/finbert_...


### Preprocessing the headline df

In [ ]:
# Create a mapping of Date to general file_path
general_paths = headline_df[headline_df['stock'] == 'general'][['Date', 'file_path']].set_index('Date')['file_path'].reset_index().rename(columns={'Date': 'date', 'file_path': 'file_path_general'})

# Only get the specific filepath
headline_df = headline_df[['Date', 'stock', 'file_path']]

headline_df['Date'] = pd.to_datetime(headline_df['Date'])

# Convert Date column to datetime if not already
headline_df['Date'] = pd.to_datetime(headline_df['Date'])

# Define date range
start_date = '2010-01-06'
end_date = '2020-07-19'

# Filter the DataFrame
headline_df = headline_df[(headline_df['Date'] >= start_date) & (headline_df['Date'] <= end_date)]

headline_df.head()

,Date,stock,file_path
225,2010-01-06,AXP,/content/drive/MyDrive/financial_data/finbert_...
226,2010-01-06,MA,/content/drive/MyDrive/financial_data/finbert_...
227,2010-01-06,general,/content/drive/MyDrive/financial_data/finbert_...
228,2010-01-07,ACN,/content/drive/MyDrive/financial_data/finbert_...
229,2010-01-07,BLK,/content/drive/MyDrive/financial_data/finbert_...


In [ ]:
# --- Data Preparation ---

# 1. Convert date columns to datetime objects for proper merging and filtering
headline_df['Date'] = pd.to_datetime(headline_df['Date'])
ohlcv_df['date'] = pd.to_datetime(ohlcv_df['date'])
is_easy_df['date'] = pd.to_datetime(is_easy_df['date'])
general_paths['date'] = pd.to_datetime(general_paths['date'])
# 2. Rename headline date column to match the others
headline_df = headline_df.rename(columns={'Date': 'date'})

# 3. Optional: Drop unnecessary columns from headline_df
headline_df = headline_df[['date', 'stock', 'file_path']]

# --- Merging Process ---

# Step 1: Merge ohlcv_df and is_easy_df on 'date' and 'stock'
# Using 'inner' merge keeps only rows where a stock exists on a specific date in BOTH dataframes
merged_ohlcv_easy = pd.merge(ohlcv_df, is_easy_df, on=['date', 'stock'], how='inner')

# Step 2: Merge the result with headline_df on 'date'
# Using 'inner' merge ensures that we only keep data for dates present in headline_df.
# This automatically restricts the final dataframe to the desired time range.
merged_df = pd.merge(merged_ohlcv_easy, headline_df, on=['date', 'stock'], how='left')

# Get the general paths
merged_df = pd.merge(merged_df, general_paths, on=['date'], how='left')

# 3. Determine the minimum and maximum dates from headline_df
min_date = headline_df['date'].min()
max_date = headline_df['date'].max()
final_merged_df = merged_df[(merged_df['date'] >= min_date) & (merged_df['date'] <= max_date)]

# 4. Confirmed the stocks are within the list
stocks = [
    "PLTR", "TSLA", "KKR", "AMD", "NVDA", "BX", "BA", "AMAT", "LRCX", "C",
    "DIS", "BKNG", "ISRG", "GS", "UBER", "MS", "BAC", "ADBE", "CRM", "BLK",
    "NFLX", "KLAC", "QCOM", "INTU", "AXP", "ACN", "GE", "SPGI", "AAPL", "META",
    "COP", "MU", "WFC", "CRWD", "AMZN", "PANW", "PLD", "JPM", "CAT", "LOW",
    "MA", "CVX", "ANET", "INTC", "UNP", "HD", "ORCL", "HON", "ETN", "ADI"
]

final_merged_df = final_merged_df[final_merged_df['stock'].isin(stocks)]


In [ ]:
final_merged_df[final_merged_df['stock'] == "AAPL"]

,date,stock,open,high,low,close,volume,easy_indicator,file_path,file_path_general
2,2010-01-06,AAPL,6.451467,6.477047,6.342227,6.348848,552160000,0,NaN,/content/drive/MyDrive/financial_data/finbert_...
3,2010-01-07,AAPL,6.372321,6.379844,6.291068,6.337111,477131200,1,NaN,/content/drive/MyDrive/financial_data/finbert_...
4,2010-01-08,AAPL,6.328683,6.379843,6.291368,6.379241,447610800,0,NaN,NaN
5,2010-01-11,AAPL,6.403918,6.409937,6.273011,6.322967,462229600,0,NaN,/content/drive/MyDrive/financial_data/finbert_...
6,2010-01-12,AAPL,6.295279,6.312734,6.211920,6.251041,594459600,1,NaN,/content/drive/MyDrive/financial_data/finbert_...
...,...,...,...,...,...,...,...,...,...,...
2511,2019-12-24,AAPL,68.924708,68.973132,68.496186,68.823021,48478800,0,NaN,/content/drive/MyDrive/financial_data/finbert_...
2512,2019-12-26,AAPL,68.956189,70.205449,68.927137,70.188499,93121200,0,NaN,/content/drive/MyDrive/financial_data/finbert_...
2513,2019-12-27,AAPL,70.481437,71.171436,69.755124,70.161858,146266000,0,NaN,/content/drive/MyDrive/financial_data/finbert_...
2514,2019-12-30,AAPL,70.079558,70.861558,69.053038,70.578293,144114400,0,NaN,/content/drive/MyDrive/financial_data/finbert_...


### Build the relevant datasets

In [ ]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import pandas as pd
from tqdm import tqdm
from collections import defaultdict
from transformers import PatchTSMixerConfig, PatchTSMixerForTimeSeriesClassification
from torch import nn, optim
from sklearn.metrics import accuracy_score, f1_score

class StockClassificationDataset(Dataset):
    def __init__(self, df, sequence_length=252, forecast_horizon=1,
                 feature_cols=['open', 'high', 'low', 'close', 'volume'],
                 sentiment_dim=768, output_dir='./embedding_cache', dataset_type='train'):
        """
        Initialize the dataset with preloaded embeddings, saving/loading caches to/from output_dir.

        Parameters:
        - df: DataFrame with columns: date, stock, open, high, low, close, volume, easy_indicator, file_path, file_path_general
        - sequence_length: Number of days in each sequence (default: 252)
        - forecast_horizon: Number of days ahead to predict (default: 1)
        - feature_cols: List of OHLCV feature columns
        - sentiment_dim: Dimensionality of sentiment embeddings (default: 768)
        - output_dir: Directory to save/load embedding cache files (default: './embedding_cache')
        - dataset_type: 'train' or 'test' to distinguish cache files (default: 'train')
        """
        assert sequence_length >= forecast_horizon, "Sequence length must be at least as long as forecast_horizon."

        self.sequence_length = sequence_length
        self.forecast_horizon = forecast_horizon
        self.feature_cols = feature_cols
        self.sentiment_dim = sentiment_dim
        self.output_dir = output_dir
        self.dataset_type = dataset_type

        # Create output directory if it doesn't exist
        os.makedirs(output_dir, exist_ok=True)

        # Preprocess DataFrame: remove duplicates and sort
        df['date'] = pd.to_datetime(df['date'])
        self.df = df.drop_duplicates(subset=['stock', 'date'], keep='first').sort_values(['stock', 'date']).reset_index(drop=True)

        # Store unique dates and stock data
        self.unique_dates = sorted(self.df['date'].unique())
        self.stock_data = {stock: group.reset_index(drop=True) for stock, group in self.df.groupby('stock')}

        # Define cache file paths with dataset_type suffix
        specific_cache_path = os.path.join(output_dir, f'specific_embeddings_{dataset_type}.npz')
        general_cache_path = os.path.join(output_dir, f'general_embeddings_{dataset_type}.npz')

        # Try to load embedding caches
        self.specific_embedding_cache = {}
        self.general_embedding_cache = {}
        caches_loaded = False

        if os.path.exists(specific_cache_path) and os.path.exists(general_cache_path):
            try:
                # Load specific embeddings
                specific_npz = np.load(specific_cache_path, allow_pickle=True)
                self.specific_embedding_cache = {key: specific_npz[key] for key in specific_npz.files}

                # Load general embeddings
                general_npz = np.load(general_cache_path, allow_pickle=True)
                self.general_embedding_cache = {key: general_npz[key] for key in general_npz.files}

                print(f"Loaded embedding caches from {specific_cache_path} and {general_cache_path}")
                caches_loaded = True
            except Exception as e:
                print(f"Error loading caches: {e}. Rebuilding caches...")

        if not caches_loaded:
            # Preload stock-specific embeddings (file_path)
            unique_specific_paths = self.df['file_path'].dropna().unique()
            for fp in tqdm(unique_specific_paths, desc=f"Preloading stock-specific embeddings ({dataset_type})"):
                if os.path.exists(fp):
                    emb = torch.load(fp)
                    if isinstance(emb, torch.Tensor):
                        emb = emb.cpu().numpy()
                    self.specific_embedding_cache[fp] = emb

            # Preload general embeddings (file_path_general)
            unique_general_paths = self.df['file_path_general'].dropna().unique()
            for fp in tqdm(unique_general_paths, desc=f"Preloading general embeddings ({dataset_type})"):
                if os.path.exists(fp):
                    emb = torch.load(fp)
                    if isinstance(emb, torch.Tensor):
                        emb = emb.cpu().numpy()
                    self.general_embedding_cache[fp] = emb

            # Save caches to .npz files
            try:
                np.savez_compressed(specific_cache_path, **self.specific_embedding_cache)
                np.savez_compressed(general_cache_path, **self.general_embedding_cache)
                print(f"Saved embedding caches to {specific_cache_path} and {general_cache_path}")
            except Exception as e:
                print(f"Error saving caches: {e}")

        # Build valid sequence indices
        self.valid_sequences = []
        for forecast_date in tqdm(self.unique_dates, desc="Processing Dates"):
            forecast_idx = self.unique_dates.index(forecast_date)
            if forecast_idx < sequence_length:
                continue  # Skip dates without enough prior data

            start_date = self.unique_dates[forecast_idx - sequence_length]
            for stock, stock_df in self.stock_data.items():
                seq_df = stock_df[(stock_df['date'] >= start_date) & (stock_df['date'] < forecast_date)]
                forecast_row = stock_df[stock_df['date'] == forecast_date]

                if (len(seq_df) == self.sequence_length and
                    not forecast_row.empty and
                    forecast_row.index[0] + self.forecast_horizon - 1 < len(stock_df)):
                    stock_idx = forecast_row.index[0]
                    self.valid_sequences.append({
                        'forecast_date': forecast_date,
                        'stock': stock,
                        'forecast_idx': stock_idx
                    })

        # Sort sequences by forecast date for batching
        self.valid_sequences.sort(key=lambda x: x['forecast_date'])

    def __len__(self):
        return len(self.valid_sequences)

    def __getitem__(self, idx):
        """Load data on-demand for a specific sequence."""
        seq_info = self.valid_sequences[idx]
        forecast_date = seq_info['forecast_date']
        stock = seq_info['stock']
        forecast_idx = seq_info['forecast_idx']
        stock_df = self.stock_data[stock]

        # Extract sequence
        start_idx = forecast_idx - self.sequence_length
        seq_df = stock_df.iloc[start_idx:forecast_idx]

        # Load OHLCV features
        seq_features = seq_df[self.feature_cols].values  # Shape: (sequence_length, num_features)
        scaler = StandardScaler()
        seq_features = scaler.fit_transform(seq_features)

        # Load sentiment embeddings from specific_embedding_cache
        seq_file_paths = seq_df['file_path'].values
        seq_sentiments = []
        for fp in seq_file_paths:
            emb = self.specific_embedding_cache.get(fp, np.zeros(self.sentiment_dim, dtype=np.float32))
            seq_sentiments.append(emb)
        seq_sentiments = np.stack(seq_sentiments)  # Shape: (sequence_length, sentiment_dim)

        # Load general sentiment embeddings from general_embedding_cache
        seq_file_paths_general = seq_df['file_path_general'].values
        seq_general_sentiments = []
        for fp in seq_file_paths_general:
            emb = self.general_embedding_cache.get(fp, np.zeros(self.sentiment_dim, dtype=np.float32))
            seq_general_sentiments.append(emb)
        seq_general_sentiments = np.stack(seq_general_sentiments)  # Shape: (sequence_length, sentiment_dim)

        # Compute target
        start_close = seq_df.iloc[-1]['close']
        target_close = stock_df.iloc[forecast_idx + self.forecast_horizon - 1]['close']
        pct_change = (target_close - start_close) / start_close * 100
        target = 2 if pct_change > 0.2 else (0 if pct_change < -0.2 else 1)  # Rise, Drop, Neutral
        easy_mask = stock_df.iloc[forecast_idx + self.forecast_horizon - 1]['easy_indicator']

        return {
            'features': torch.tensor(seq_features, dtype=torch.float),
            'sentiment': torch.tensor(seq_sentiments, dtype=torch.float),
            'general_sentiment': torch.tensor(seq_general_sentiments, dtype=torch.float),
            'target': torch.tensor(target, dtype=torch.long),
            'easy_mask': torch.tensor(easy_mask, dtype=torch.float),
            'forecast_date': str(forecast_date)
        }

# Custom collate function
def collate_fn(batch):
    if not batch:
        return None
    keys = batch[0].keys()
    return {key: torch.stack([item[key] for item in batch]) for key in keys}

def train_classification(model, dataloader, epochs=5, lr=1e-3,
                         use_sentiment=True, use_general=True, use_mask=True, proj_dim=5, device='cuda'):
    """
    Train a classification model using a DataLoader that yields batches with:
      - 'features': (batch, seq_len, 5)
      - 'sentiment': (batch, seq_len, 768) - stock-specific sentiment
      - 'general_sentiment': (batch, seq_len, 768) - general sentiment
      - 'target': (batch,) with values in {0,1,2}
      - 'easy_mask': (batch,) with 1 for easy, 0 otherwise.

    If use_sentiment is True, a shared projection layer (768 -> proj_dim) is applied to sentiment
    embeddings. If use_general is also True, both sentiment and general_sentiment are used,
    resulting in (batch, seq_len, 5 + proj_dim + proj_dim). If use_general is False, only
    sentiment is used, resulting in (batch, seq_len, 5 + proj_dim).

    For the first 20% of training steps, the loss includes all samples (easy and non-easy).
    After that, if use_mask is True, the loss is computed only on non-easy samples (easy_mask == 0).
    Reports training loss, accuracy, and weighted F1 score for each epoch.
    """
    model.to(device)
    model.train()

    if use_sentiment:
        projection = nn.Linear(768, proj_dim).to(device)  # Shared projection for both sentiments
        optimizer = optim.Adam(list(model.parameters()) + list(projection.parameters()), lr=lr)
    else:
        projection = None
        optimizer = optim.Adam(model.parameters(), lr=lr)

    criterion = nn.CrossEntropyLoss()

    # Compute total steps and unmasked steps (first 20%)
    total_steps = len(dataloader) * epochs
    unmasked_steps = int(0.2 * total_steps)
    current_step = 0

    for epoch in range(epochs):
        epoch_loss = 0.0
        all_preds = []
        all_labels = []
        pbar = tqdm(dataloader, desc=f"Epoch {epoch}")
        for batch in pbar:
            features = batch['features'].to(device)          # shape: (B, seq_len, 5)
            sentiment = batch['sentiment'].to(device)        # shape: (B, seq_len, 768)
            general_sentiment = batch['general_sentiment'].to(device)  # shape: (B, seq_len, 768)
            target = batch['target'].to(device)              # shape: (B,)
            easy_mask = batch['easy_mask'].to(device)        # shape: (B,)

            if use_sentiment:
                proj_sent = projection(sentiment)            # (B, seq_len, proj_dim)
                if use_general:
                    proj_general_sent = projection(general_sentiment)  # (B, seq_len, proj_dim)
                    combined = torch.cat((features, proj_sent, proj_general_sent), dim=-1)  # (B, seq_len, 5+proj_dim+proj_dim)
                else:
                    combined = torch.cat((features, proj_sent), dim=-1)  # (B, seq_len, 5+proj_dim)
            else:
                combined = features  # (B, seq_len, 5)

            optimizer.zero_grad()
            outputs = model(past_values=combined)
            logits = outputs.prediction_outputs  # shape: (B, num_classes)

            loss_raw = criterion(logits, target)  # shape: (B,)
            if current_step < unmasked_steps:
                loss = loss_raw.mean()  # Include all samples
            else:
                if use_mask:
                    mask = (1.0 - easy_mask)  # shape: (B,)
                    loss = (loss_raw * mask).sum() / (mask.sum() + 1e-8)
                else:
                    loss = loss_raw.mean()

            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

            preds = torch.argmax(logits, dim=-1)
            all_preds.append(preds.cpu().numpy())
            all_labels.append(target.cpu().numpy())

            current_step += 1
            pbar.set_postfix(loss=loss.item())

        all_preds = np.concatenate(all_preds)
        all_labels = np.concatenate(all_labels)
        epoch_acc = accuracy_score(all_labels, all_preds)
        epoch_f1 = f1_score(all_labels, all_preds, average='weighted')
        print(f"Epoch {epoch}: Loss = {epoch_loss/len(dataloader):.4f}, "
              f"Accuracy = {epoch_acc:.4f}, F1 Score = {epoch_f1:.4f}")

    if use_sentiment:
        torch.save(projection.state_dict(), f"/content/drive/MyDrive/financial_data/model_v2/projection_masked_{use_mask}_general_{use_general}_forecast_20_1_5_20.pth")

def evaluate_classification(model, dataloader, use_sentiment, use_general, proj_dim=5, device='cuda'):
    """
    Evaluate a classification model on the test set. Returns (accuracy, weighted_f1).
    """
    model.to(device)
    model.eval()

    if use_sentiment:
        projection = nn.Linear(768, proj_dim).to(device)
    else:
        projection = None

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            features = batch['features'].to(device)
            sentiment = batch['sentiment'].to(device)
            general_sentiment = batch['general_sentiment'].to(device)
            labels = batch['target'].to(device)

            if use_sentiment:
                proj_sent = projection(sentiment)
                if use_general:
                    proj_general_sent = projection(general_sentiment)
                    combined = torch.cat((features, proj_sent, proj_general_sent), dim=-1)
                else:
                    combined = torch.cat((features, proj_sent), dim=-1)
            else:
                combined = features

            outputs = model(past_values=combined)
            logits = outputs.prediction_outputs
            preds = torch.argmax(logits, dim=-1)
            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted')
    return acc, f1

def run_ablation_experiment(train_dataloader, test_dataloader, use_sentiment, use_general, use_mask,
                            epochs=5, lr=1e-3, proj_dim=5, device='cuda', exp_name="default"):
    # Set num_input_channels: 15 if both sentiment and general, 10 if only sentiment, 5 if no sentiment
    num_input_channels = 15 if use_sentiment and use_general else (10 if use_sentiment else 5)
    config = PatchTSMixerConfig(context_length=252, patch_length=12, patch_stride=12,
                                num_input_channels=num_input_channels, num_targets=3,
                                d_model=32, num_layers=8)
    model = PatchTSMixerForTimeSeriesClassification(config)

    # Train
    train_classification(model, train_dataloader, epochs=epochs, lr=lr,
                         use_sentiment=use_sentiment, use_general=use_general, use_mask=use_mask,
                         proj_dim=proj_dim, device=device)

    # Save model state dictionary
    save_dir = "/content/drive/MyDrive/financial_data/model_v2"
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, f"model_{exp_name}_forecast_20_1_5_20.pth")
    torch.save(model.state_dict(), save_path)
    print(f"Model saved to {save_path}")

    # Evaluate on test set
    accuracy, f1 = evaluate_classification(model, test_dataloader, use_sentiment=use_sentiment,
                                          use_general=use_general, proj_dim=proj_dim, device=device)
    return accuracy, f1

def main():
    # Prepare train and test DataLoaders
    split_date = pd.Timestamp('2017-01-01')
    train_df = final_merged_df[final_merged_df['date'] < split_date].reset_index(drop=True)
    test_df = final_merged_df[final_merged_df['date'] >= split_date].reset_index(drop=True)

    forecast_horizon = 20
    output_dir = "/content/drive/MyDrive/financial_data/embedding_cache"
    train_dataset = StockClassificationDataset(train_df, sequence_length=252, forecast_horizon=forecast_horizon,
                                               output_dir=output_dir, dataset_type='train')
    test_dataset = StockClassificationDataset(test_df, sequence_length=252, forecast_horizon=forecast_horizon,
                                              output_dir=output_dir, dataset_type='test')

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    print("Train samples:", len(train_dataset))
    print("Test samples:", len(test_dataset))

    ablations = [
        {"name": "With_Sentiment_With_General_Mask", "use_sentiment": True, "use_general": True, "use_mask": True},
        {"name": "With_Sentiment_With_General_NoMask", "use_sentiment": True, "use_general": True, "use_mask": False},
        {"name": "With_Sentiment_No_General_Mask", "use_sentiment": True, "use_general": False, "use_mask": True},
        {"name": "With_Sentiment_No_General_NoMask", "use_sentiment": True, "use_general": False, "use_mask": False},
        {"name": "Without_Sentiment_Mask", "use_sentiment": False, "use_general": False, "use_mask": True},
        {"name": "Without_Sentiment_NoMask", "use_sentiment": False, "use_general": False, "use_mask": False},
    ]

    results = []
    for ab in ablations:
        print(f"=== Running Experiment: {ab['name']} ===")
        acc, f1 = run_ablation_experiment(train_loader, test_loader,
                                          use_sentiment=ab["use_sentiment"],
                                          use_general=ab["use_general"],
                                          use_mask=ab["use_mask"],
                                          epochs=20, lr=1e-4, proj_dim=5, device='cuda', exp_name=ab["name"])
        results.append({"Experiment": ab["name"], "Accuracy": acc, "F1 Score": f1})

    print("\nAblation Study Results:")
    print("{:<30} {:<15} {:<15}".format("Experiment", "Accuracy", "F1 Score"))
    for res in results:
        print("{:<30} {:<15.4f} {:<15.4f}".format(res["Experiment"], res["Accuracy"], res["F1 Score"]))

if __name__ == "__main__":
    main()

Loaded embedding caches from /content/drive/MyDrive/financial_data/embedding_cache/specific_embeddings_train.npz and /content/drive/MyDrive/financial_data/embedding_cache/general_embeddings_train.npz


Processing Dates: 100%|██████████| 1760/1760 [00:54<00:00, 32.30it/s]


Loaded embedding caches from /content/drive/MyDrive/financial_data/embedding_cache/specific_embeddings_test.npz and /content/drive/MyDrive/financial_data/embedding_cache/general_embeddings_test.npz


Processing Dates: 100%|██████████| 754/754 [00:18<00:00, 41.02it/s]


Train samples: 67284
Test samples: 22701
=== Running Experiment: With_Sentiment_With_General_Mask ===


...

Epoch 18: Loss = 0.7439, Accuracy = 0.6086, F1 Score = 0.5736


Epoch 19: 100%|██████████| 2103/2103 [04:28<00:00,  7.83it/s, loss=0.98]


Epoch 19: Loss = 0.7413, Accuracy = 0.6130, F1 Score = 0.5804
Model saved to /content/drive/MyDrive/financial_data/model_v2/model_Without_Sentiment_NoMask_forecast_20_1_5_20.pth


Evaluating: 100%|██████████| 710/710 [01:17<00:00,  9.20it/s]


Ablation Study Results:
Experiment                     Accuracy        F1 Score       
With_Sentiment_With_General_Mask 0.4828          0.4736         
With_Sentiment_With_General_NoMask 0.4747          0.4639         
With_Sentiment_No_General_Mask 0.5242          0.4416         
With_Sentiment_No_General_NoMask 0.5533          0.4153         
Without_Sentiment_Mask         0.5494          0.4068         
Without_Sentiment_NoMask       0.5566          0.4119         
